# Explicação de `main.py`: um sítio de Brod-Combes com um banho em estado de Fock

Este notebook percorre `main.py` linha por linha. Para cada variável e cada expressão, ele diz o que ela representa e
escreve a fórmula em LaTeX. As células de código reproduzem o cálculo do script. A única diferença é que **não salvam**
as figuras, para não sobrescrever `figura1_rapida_lenta.*`.

**O que o script calcula.** Temos um átomo de dois níveis acoplado a um guia de onda com taxa $\gamma$. Esse átomo
também está acoplado a **um único** modo harmônico (a "mola", ou banho) de frequência $\omega$, que começa no estado
de Fock $|n\rangle$. Um fóton de frequência $\nu_a$ incide sobre o átomo. O script calcula a probabilidade de o fóton
sair **"limpo"**, ou seja, de o espalhamento ser elástico e a mola terminar no mesmo estado $|n\rangle$:

$$
P_{\text{limpo}}(\nu_a) = \big|t_{nn}(\nu_a)\big|^2 .
$$

## 1. Importações

| Nome | Origem | Uso |
|---|---|---|
| `np` | `numpy` | arrays, `sqrt`, `exp`, `linspace` |
| `eval_genlaguerre` | `scipy.special` | polinômio de Laguerre generalizado $L_n^{(\alpha)}(x)$ |
| `factorial` | `scipy.special` | fatorial $n!$ (em ponto flutuante) |
| `plt` | `matplotlib.pyplot` | figuras |

In [ ]:
import numpy as np
from scipy.special import eval_genlaguerre, factorial
import matplotlib.pyplot as plt

## 2. Parâmetros do átomo

Todas as frequências são medidas em unidades de $\gamma$. Por isso `gam = 1`.

| Variável | Símbolo | Significado |
|---|---|---|
| `gam` | $\gamma$ | taxa de emissão do átomo no guia. Fixa a unidade, e a meia-largura da linha é $\gamma/2$ |
| `Delta` | $\Delta$ | dessintonia do átomo, isto é, onde a linha estaria sem o banho |

In [ ]:
gam   = 1.0     # gamma: taxa de emissao do atomo no guia (define a unidade)
Delta = 0.0     # Delta: dessintonia do atomo (onde fica a linha sem banho)

## 3. Parâmetros das molas (banhos)

### Fator de Huang-Rhys $S$

O acoplamento átomo–mola desloca o equilíbrio do oscilador quando o átomo está excitado. O deslocamento é medido
pelo parâmetro adimensional

$$
\beta = \frac{2g}{\omega},
$$

e o **fator de Huang-Rhys**, que dá o "tamanho do puxão", é

$$
S = \beta^2 = \left(\frac{2g}{\omega}\right)^2 .
$$

O script fixa $S = 0.25$ e muda apenas $\omega$. Assim a comparação isola o efeito da *velocidade* da mola. Para isso,
o acoplamento $g$ é calculado a partir de $S$ e $\omega$ (seção 7):

$$
g = \frac{\sqrt{S}\,\omega}{2}.
$$

### Lista `casos`

Cada entrada é uma tupla `(titulo, omega, K)`:

| Caso | $\omega/\gamma$ | $K$ | Regime |
|---|---|---|---|
| Mola rápida | $3.0$ | $60$ | $\omega \gg \gamma$: as bandas laterais ficam separadas |
| Mola lenta | $0.3$ | $40$ | $\omega \ll \gamma$: as bandas laterais caem dentro da linha |

- `K` é o **corte numérico**: a soma sobre os níveis $k$ do oscilador deslocado usa só $k = 0, 1, \dots, K-1$.
  Com $S = 0.25$, o fator de Franck-Condon decai muito rápido em $k$, então esses cortes sobram (ver a verificação na seção 8).

### Demais variáveis

| Variável | Significado |
|---|---|
| `ns` | estados de Fock iniciais da mola: $n \in \{0, 1, 2\}$ |
| `cores` | cor da curva de cada $n$ (preto, azul, vermelho) |
| `nus` | eixo $x$: frequência do fóton incidente, $\nu_a/\gamma \in [-5, 5]$, com 801 pontos |

In [ ]:
S = 0.25                                    # fator de Huang-Rhys
casos = [
    # (titulo,        omega, K)
    ("Mola rapida",   3.0,   60),            # omega >> gamma: bandas laterais separadas
    ("Mola lenta",    0.3,   40),            # omega << gamma: bandas dentro da linha
]
ns    = [0, 1, 2]                           # estados de Fock iniciais da mola
cores = {0: 'k', 1: 'b', 2: 'r'}            # cor de cada curva n
nus   = np.linspace(-5, 5, 801)             # eixo x: cor do foton que entra, nu_a/gamma

## 4. Fator de Franck-Condon — `FC(k, n, beta)`

O fator de Franck-Condon é o elemento de matriz do **operador deslocamento**
$D(\beta) = e^{\beta(a^\dagger - a)}$ entre estados de Fock:

$$
F_{kn} \equiv \langle k | D(\beta) | n \rangle .
$$

Ele dá a amplitude de a mola passar de $|n\rangle$ (no poço do estado fundamental do átomo) para o nível $|k\rangle$
do poço deslocado (átomo excitado).

**Caso $k \ge n$.** Para $\beta$ real, este é o ramo calculado diretamente no código:

$$
F_{kn} = \sqrt{\frac{n!}{k!}}\;\beta^{\,k-n}\;e^{-\beta^2/2}\;L_n^{(k-n)}\!\left(\beta^2\right),
$$

em que $L_n^{(\alpha)}$ é o polinômio de Laguerre generalizado (`eval_genlaguerre(n, k-n, beta**2)`).

**Caso $k < n$.** Como $D(\beta)^\dagger = D(-\beta)$ e $\beta$ é real, vale a simetria

$$
F_{kn} = (-1)^{\,n-k}\,F_{nk},
$$

e o código chama a si mesmo com os índices trocados.

**Propriedade útil (regra de soma).** Como $D$ é unitário e $\{|k\rangle\}$ é completo,

$$
\sum_{k=0}^{\infty} F_{kn}^2 = 1 .
$$

Para $n = 0$, $F_{k0}^2 = e^{-S} S^k / k!$ é uma distribuição de Poisson de média $S$.

In [ ]:
def FC(k, n, beta):
    if k >= n:
        return (np.sqrt(factorial(n) / factorial(k)) * beta**(k - n)
                * np.exp(-beta**2 / 2) * eval_genlaguerre(n, k - n, beta**2))
    return (-1)**(n - k) * FC(n, k, beta)   # simetria para k < n

## 5. Amplitude de espalhamento — `t_mn(m, n, nu, gam, Delta, w, g, K)`

A amplitude para o fóton passar pelo átomo levando a mola de $|n\rangle$ para $|m\rangle$ é

$$
t_{mn}(\nu_a) = \delta_{mn} \;-\; \gamma \sum_{k=0}^{K-1}
\frac{F_{km}\,F_{kn}}
{\dfrac{\gamma}{2} + i\left(\Delta - \dfrac{4g^2}{\omega} + (k-n)\,\omega - \nu_a\right)} .
$$

Cada parte da fórmula:

| Termo | Variável no código | Significado |
|---|---|---|
| $\delta_{mn}$ | `1.0 if m == n else 0.0` | parte do fóton que passa sem interagir. Só existe no canal elástico |
| $\gamma$ (fora da soma) | `gam` | intensidade do acoplamento ao guia |
| $k$ | `k` | nível intermediário do oscilador **deslocado**, com o átomo excitado |
| $F_{kn}$ | `FC(k, n, beta)` | amplitude de absorção: $\lvert g,n\rangle \to \lvert e,\tilde k\rangle$ |
| $F_{km}$ | `FC(k, m, beta)` | amplitude de emissão: $\lvert e,\tilde k\rangle \to \lvert g,m\rangle$ |
| $\gamma/2$ | `gam / 2` | meia-largura da ressonância |
| $4g^2/\omega$ | `4 * g**2 / w` | **deslocamento polarônico** (reorganização), igual a $S\omega$ |
| $(k-n)\,\omega$ | `(k - n) * w` | energia de vibração ganha ($k>n$) ou perdida ($k<n$) na absorção |
| $\nu_a$ | `nu` | frequência do fóton incidente |
| denominador | `den` | `gam / 2 + 1j * (Delta - 4*g**2/w + (k-n)*w - nu)` |

Duas identidades:

$$
\frac{4g^2}{\omega} = \omega\beta^2 = S\,\omega ,
$$

e cada termo da soma gera uma ressonância (um "vale" em $|t_{nn}|^2$) em

$$
\nu_a^{(k)} = \Delta - S\omega + (k-n)\,\omega .
$$

O termo $k = n$ é a **linha de zero fônon** (ZPL). Os termos $k \neq n$ são as **bandas laterais** de fônon.

**Limite sem banho ($g = 0$).** Nesse caso $F_{kn} = \delta_{kn}$ e

$$
t_{nn} = 1 - \frac{\gamma}{\gamma/2 - i(\nu_a - \Delta)}
= -\,\frac{\gamma/2 + i(\nu_a - \Delta)}{\gamma/2 - i(\nu_a - \Delta)},
\qquad |t_{nn}|^2 = 1 .
$$

Sem o banho, o fóton sempre sai "limpo" e só ganha uma fase. Logo, **qualquer queda de $|t_{nn}|^2$ abaixo de 1 vem do
espalhamento inelástico**, em que o fóton deixa ou retira quanta da mola.

In [ ]:
def t_mn(m, n, nu, gam, Delta, w, g, K):
    beta = 2 * g / w
    s = 0.0
    for k in range(K):
        den = gam / 2 + 1j * (Delta - 4 * g**2 / w + (k - n) * w - nu)
        s += FC(k, m, beta) * FC(k, n, beta) / den
    return (1.0 if m == n else 0.0) - gam * s

## 6. Verificações numéricas (não estão em `main.py`)

Três testes rápidos de consistência:

1. **Regra de soma de Franck-Condon:** $\sum_{k<K} F_{kn}^2 \approx 1$. Mostra que o corte `K` é suficiente.
2. **Unitariedade:** $\sum_m |t_{mn}(\nu_a)|^2 = 1$. A probabilidade se divide entre o canal limpo ($m = n$) e os canais
   inelásticos ($m \neq n$), sem perdas. Então $1 - |t_{nn}|^2$ é exatamente a probabilidade de o fóton sair "sujo".
3. **Limite $g = 0$:** $|t_{nn}| = 1$.

In [ ]:
nu_teste = 0.7
for titulo, w, K in casos:
    g = np.sqrt(S) * w / 2
    beta = 2 * g / w
    for n in ns:
        soma_FC = sum(FC(k, n, beta)**2 for k in range(K))
        soma_t  = sum(abs(t_mn(m, n, nu_teste, gam, Delta, w, g, K))**2 for m in range(K - 5))
        print(f'{titulo:12s} n={n}:  sum_k F_kn^2 = {soma_FC:.12f}   sum_m |t_mn|^2 = {soma_t:.12f}')

print('g = 0  ->  |t_00| =', abs(t_mn(0, 0, nu_teste, gam, Delta, 3.0, 0.0, 5)))

## 7. Laço principal e figura

Em cada painel (caso), o script faz o seguinte:

1. Calcula o acoplamento que realiza o $S$ escolhido:
   $$ g = \frac{\sqrt{S}\,\omega}{2} \qquad (\texttt{g = np.sqrt(S) * w / 2}). $$
2. Calcula a posição da linha de zero fônon, já deslocada pelo polaron:
   $$ \nu_{\text{ZPL}} = \Delta - \frac{4g^2}{\omega} = \Delta - S\omega \qquad (\texttt{zpl}). $$
3. Para cada $n \in$ `ns`, calcula a curva
   $$ \texttt{limpo}(\nu_a) = |t_{nn}(\nu_a)|^2 $$
   nos 801 pontos de `nus` e imprime o mínimo.
4. Desenha linhas pontilhadas nas posições dos vales
   $$ x_j = \nu_{\text{ZPL}} + j\,\omega, \qquad j = -\lfloor 5/\omega \rfloor - 1,\ \dots,\ \lfloor 5/\omega \rfloor + 1, $$
   mantendo só as que caem em $[-5, 5]$. O índice $j$ corresponde a $k - n$ na fórmula de $t_{mn}$.

Valores numéricos que resultam disso:

| Caso | $\omega$ | $g = \sqrt{S}\,\omega/2$ | $\beta = 2g/\omega$ | $S\omega = 4g^2/\omega$ (ZPL em $-S\omega$) |
|---|---|---|---|---|
| Mola rápida | $3.0$ | $0.75$ | $0.5$ | $0.75$ |
| Mola lenta | $0.3$ | $0.075$ | $0.5$ | $0.075$ |

**Como ler o resultado.** Na mola rápida ($\omega = 3\gamma \gg \gamma$), as ressonâncias $k - n = 0, \pm 1, \dots$ ficam bem
separadas, e aparecem vales distintos: a ZPL e as bandas laterais. Na mola lenta ($\omega = 0.3\gamma \ll \gamma$), as
ressonâncias ficam mais próximas que a largura $\gamma/2$ e se fundem num único vale largo. Com $n > 0$ abre-se também o canal de *perder* quanta ($k < n$), e os fatores $F_{kn}$ mudam de forma. Na mola lenta, isso deixa o vale
mais fundo a cada $n$ (mínimos $\approx 0.75,\ 0.44,\ 0.25$). Na mola rápida, a dependência não é monótona (mínimos $\approx 0.31,\ 0.01,\ 0.10$
para $n = 0, 1, 2$), Isso se entende por uma estimativa. Quando as linhas estão isoladas, perto da ZPL vale
$t_{nn} \approx 1 - 2F_{nn}^2$, de modo que o vale é mais fundo quando $F_{nn}^2 \approx 1/2$. Temos $F_{00}^2 \approx 0.78$,
que dá $|1-2F^2|^2 \approx 0.31$, e $F_{11}^2 \approx 0.44$, que dá $\approx 0.015$. Já para $n = 2$, $F_{22}^2 \approx 0.22$, e o
mínimo global deixa de estar na ZPL: ele cai na banda lateral $k - n = +1$, em $\nu_a \approx 2.2$.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 4), sharey=True)

for ax, (titulo, w, K) in zip(axs, casos):
    g = np.sqrt(S) * w / 2                  # acoplamento que da o S escolhido
    zpl = Delta - 4 * g**2 / w              # vale do meio (deslocado pelo polaron)

    for n in ns:
        limpo = [abs(t_mn(n, n, nu, gam, Delta, w, g, K))**2 for nu in nus]
        ax.plot(nus, limpo, cores[n], label=f'$n={n}$')
        print(f'{titulo}: n={n}, minimo de |t_nn|^2 = {min(limpo):.3f}')

    # linhas pontilhadas: posicoes dos vales nu_a = Delta - 4g^2/w + j*omega
    for j in range(-int(5 / w) - 1, int(5 / w) + 2):
        x = zpl + j * w
        if -5 <= x <= 5:
            ax.axvline(x, color='gray', lw=0.5, ls=':')
    # entrada da legenda para as linhas pontilhadas (traco mais grosso para ficar visivel)
    ax.plot([], [], color='gray', lw=1.0, ls=':',
            label=r'$\nu_a=\Delta-4g^2/\omega+(k-n)\,\omega$')

    ax.set_title(fr'{titulo}: $\omega={w:g}\gamma$, $g={g:.3f}\gamma$, $S={S}$')
    ax.set_xlabel(r'cor do foton que entra, $\nu_a/\gamma$')
    ax.set_xlim(-5, 5)
    ax.set_ylim(-0.03, 1.05)
    ax.legend(fontsize=9, loc='lower right')

axs[0].set_ylabel(r'chance de sair limpo, $|t_{nn}(\nu_a)|^2$')
plt.tight_layout()
# (main.py salva aqui figura1_rapida_lenta.svg e .pdf; omitido para nao sobrescrever)
plt.show()

## 8. Resumo das expressões

$$
\begin{aligned}
\beta &= \frac{2g}{\omega}, \qquad S = \beta^2, \qquad g = \frac{\sqrt{S}\,\omega}{2} \\[4pt]
F_{kn} &= \langle k|D(\beta)|n\rangle =
\begin{cases}
\sqrt{\dfrac{n!}{k!}}\,\beta^{k-n} e^{-\beta^2/2} L_n^{(k-n)}(\beta^2), & k \ge n \\[8pt]
(-1)^{n-k} F_{nk}, & k < n
\end{cases} \\[4pt]
t_{mn}(\nu_a) &= \delta_{mn} - \gamma \sum_{k=0}^{K-1}
\frac{F_{km} F_{kn}}{\gamma/2 + i\left(\Delta - 4g^2/\omega + (k-n)\omega - \nu_a\right)} \\[4pt]
P_{\text{limpo}} &= |t_{nn}(\nu_a)|^2, \qquad
\nu_a^{\text{vales}} = \Delta - S\omega + j\,\omega,\quad j \in \mathbb{Z}
\end{aligned}
$$